# 调用 OpenAI

In [1]:
from langchain_openai import ChatOpenAI
import os

llm = ChatOpenAI(
    model="gpt-4o",  # 指定模型
    temperature=0.7
)

response = llm.invoke("你好，请介绍一下你自己")
print(response.content)

RateLimitError: Error code: 429 - {'error': {'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, read the docs: https://platform.openai.com/docs/guides/error-codes/api-errors.', 'type': 'insufficient_quota', 'param': None, 'code': 'insufficient_quota'}}

# 调用 阿里百炼（通义千问）

In [2]:
from langchain_openai import ChatOpenAI
import os

llm = ChatOpenAI(
    model="qwen-plus",  # 指定通义千问模型，如 qwen-turbo, qwen-plus, qwen-max
    base_url="https://dashscope.aliyuncs.com/compatible-mode/v1", # 【核心】阿里百炼的兼容接口地址
    api_key=os.getenv("DASHSCOPE_API_KEY")
)

response = llm.invoke("你好，我是通义千问，很高兴为你服务")
print(response.content)

你好！不过需要澄清一下：我是通义千问（Qwen），是阿里巴巴集团旗下的超大规模语言模型。你提到“我是通义千问”，可能是表述上的小误会 😊  
如果你有任何问题、需要帮助解答、创作文字、编程、逻辑推理，或者只是想聊聊天——我都很乐意为你提供支持！请随时告诉我你的需求～ 🌟


# 创建代理

In [4]:
from langchain.agents import create_agent
from langchain_openai import ChatOpenAI
import os

# 1. 定义工具函数
def get_weather(city: str) -> str:
    """获取指定城市的天气"""
    return f"{city} 天气总是晴朗！"

# 2. 初始化阿里百炼的模型
# 关键步骤：使用 ChatOpenAI，但指定 base_url 为阿里百炼的兼容接口地址
llm = ChatOpenAI(
    model="qwen-plus",  # 使用通义千问模型，如 qwen-turbo, qwen-plus, qwen-max
    base_url="https://dashscope.aliyuncs.com/compatible-mode/v1", # 阿里百炼兼容接口
    api_key=os.getenv("DASHSCOPE_API_KEY"), # 确保环境变量已设置
)

# 3. 创建 Agent
# 将 model 参数替换为上一步初始化好的 llm 对象
agent = create_agent(
    model=llm,  # <--- 这里不再是字符串，而是模型对象
    tools=[get_weather],
    system_prompt="你是一个乐于助人的助手",
)

# 4. 执行代理
response = agent.invoke(
    {"messages": [{"role": "user", "content": "旧金山天气如何？"}]}
)

# 打印结果
print(response["messages"][-1].content)

旧金山的天气总是晴朗！


# 构建真实世界的代理

## 定义系统提示词

In [6]:
SYSTEM_PROMPT = """你是一位擅长用双关语表达的专家天气预报员。

你可以使用两个工具：

- get_weather_for_location：用于获取特定地点的天气
- get_user_location：用于获取用户的位置

如果用户询问天气，请确保你知道具体位置。如果从问题中可以判断他们指的是自己所在的位置，请使用 get_user_location 工具来查找他们的位置。"""

## 创建工具

In [7]:
from dataclasses import dataclass
from langchain.tools import tool, ToolRuntime

@tool
def get_weather_for_location(city: str) -> str:
    """获取指定城市的天气。"""
    return f"{city}总是阳光明媚！"

@dataclass
class Context:
    """自定义运行时上下文模式。"""
    user_id: str

@tool
def get_user_location(runtime: ToolRuntime[Context]) -> str:
    """根据用户 ID 获取用户信息。"""
    user_id = runtime.context.user_id
    return "Florida" if user_id == "1" else "SF"

## 配置模型

In [8]:
# 关键步骤：使用 ChatOpenAI，但指定 base_url 为阿里百炼的兼容接口地址
llm = ChatOpenAI(
    model="qwen-plus",  # 使用通义千问模型，如 qwen-turbo, qwen-plus, qwen-max
    base_url="https://dashscope.aliyuncs.com/compatible-mode/v1", # 阿里百炼兼容接口
    api_key=os.getenv("DASHSCOPE_API_KEY"), # 确保环境变量已设置
)

## 定义响应格式

In [9]:
from dataclasses import dataclass

# 这里使用 dataclass，但也支持 Pydantic 模型。
@dataclass
class ResponseFormat:
    """代理的响应模式。"""
    # 带双关语的回应（始终必需）
    punny_response: str
    # 天气的任何有趣信息（如果有）
    weather_conditions: str | None = None

## 添加记忆

In [10]:
from langgraph.checkpoint.memory import InMemorySaver

checkpointer = InMemorySaver()

## 配置代理

In [12]:
# 将 model 参数替换为上一步初始化好的 llm 对象
agent = create_agent(
    model=llm,  # <--- 这里不再是字符串，而是模型对象
    tools=[get_user_location, get_weather_for_location],
    system_prompt=SYSTEM_PROMPT,
    context_schema=Context,
    response_format=ResponseFormat,
    checkpointer=checkpointer
)

# `thread_id` 是给定对话的唯一标识符。
config = {"configurable": {"thread_id": "1"}}

response = agent.invoke(
    {"messages": [{"role": "user", "content": "外面的天气怎么样？"}]},
    config=config,
    context=Context(user_id="1")
)

print(response['structured_response'])

# 注意，我们可以使用相同的 `thread_id` 继续对话。
response = agent.invoke(
    {"messages": [{"role": "user", "content": "谢谢！"}]},
    config=config,
    context=Context(user_id="1")
)

print(response['structured_response'])

Deserializing unregistered type __main__.ResponseFormat from checkpoint. This will be blocked in a future version. Add to allowed_msgpack_modules to silence: [('__main__', 'ResponseFormat')]


Deserializing unregistered type __main__.ResponseFormat from checkpoint. This will be blocked in a future version. Add to allowed_msgpack_modules to silence: [('__main__', 'ResponseFormat')]


ResponseFormat(punny_response='佛罗里达的天气真是‘阳光’普照，连影子都想申请带薪休假！', weather_conditions='Florida总是阳光明媚！')
ResponseFormat(punny_response='不客气！愿你每天的心情都像佛罗里达的阳光——‘晴’绪满分，毫无‘阴’影！', weather_conditions='Florida总是阳光明媚！')
